In [0]:
from datetime import datetime
from pyspark.sql.types import StructField, StructType, TimestampType, StringType

data = [
    ("u001", datetime(2024, 1, 15, 10, 0, 0), "/home", "mobile"),
    ("u001", datetime(2024, 1, 15, 10, 5, 0), "/products", "mobile"),
    ("u001", datetime(2024, 1, 15, 10, 45, 0), "/cart", "mobile"),
    ("u001", datetime(2024, 1, 15, 10, 50, 0), "/checkout", "mobile"),
    ("u002", datetime(2024, 1, 15, 11, 0, 0), "/home", "desktop")
]

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("page_url", StringType(), True),
    StructField("device_type", StringType(), True)
])

df = spark.createDataFrame(data,schema)
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.partitionBy("user_id").orderBy("event_timestamp")

df_window = df.withColumn("previous_event_timestamp", F.lag("event_timestamp").over(window_spec))
df_gap = (df_window.withColumn("gap_minutes", 
                        F.when(F.col("previous_event_timestamp").isNotNull(), 
                        ((F.unix_timestamp("event_timestamp")-F.unix_timestamp("previous_event_timestamp")) / 60).cast("int"))
                        .otherwise(F.lit(0))
                    )
)

df_sessions = (
    df_gap.withColumn("new_session", F.when(F.col("gap_minutes")>30,1).otherwise(0))
          .withColumn("session_id", F.sum("new_session").over(window_spec)+1)
)

df_final = df_sessions.select("user_id","event_timestamp","page_url","device_type","session_id")
    
display(df_final)